# ✈️ Orquestación del ETL OpenSky con Prefect  
## Ejecución del flujo `etl_opensky_flow` y scheduling opcional

Este notebook acompaña al pipeline principal de **ETL de Tráfico Aéreo con OpenSky Network**, ya implementado siguiendo la arquitectura **Bronze → Silver → Gold** y documentado en `01_opensky_etl.ipynb`.

Mientras que el notebook anterior se centra en la **ejecución manual paso a paso** (ingesta, limpieza, enriquecimiento y visualizaciones), aquí el foco está en la **orquestación con Prefect**, utilizando la lógica definida en:

- `src/etl_utils.py` → funciones auxiliares de extracción, transformación y guardado  
- `src/etl_opensky_flow.py` → definición del flujo `etl_opensky_flow` (tasks + flow Prefect)

El flujo automatiza el recorrido completo:

- 📥 **Extracción** del snapshot dinámico desde la API pública de OpenSky  
- 🟤 **Bronze** → normalización básica y persistencia cruda en Delta Lake  
- 🥈 **Silver** → limpieza, tipificación, columnas temporales y particionado por hora  
- 🟡 **Gold** → lectura desde Silver, enriquecimiento con metadatos estáticos y guardado final  

---

## 🎯 Objetivo de este notebook

Este notebook está pensado para:

- ejecutar el flujo **`etl_opensky_flow` de forma manual**, desde Prefect  
- verificar que las tareas de cada capa se encadenan correctamente  
- revisar logs de ejecución y comportamiento general del pipeline  
- documentar una posible **ejecución programada** (cron) sin activarla por defecto

No se redefinen transformaciones ni lógica de negocio:  
simplemente se **importa el flujo ya implementado** y se lo ejecuta en un contexto controlado.

---

## 📘 Estructura de este notebook

1. **Configuración mínima e importación del flujo**
2. **Ejecución manual del pipeline `etl_opensky_flow()`**
3. **(Opcional, documentado) Ejecución programada con `serve` y cron**

Este notebook funciona como interfaz de orquestación y validación, complementando al notebook principal de ETL y preparando el proyecto para futuras integraciones con **Prefect Cloud** o despliegues en **Azure**.

## Uso del flujo definido en `src/etl_opensky_flow.py`

El flujo ETL está implementado en `src/etl_opensky_flow.py` y encapsula el pipeline completo:

- extracción desde OpenSky  
- normalización y guardado en Bronze  
- limpieza, tipificación y particionado en Silver  
- enriquecimiento y persistencia final en Gold  

Desde este notebook se puede:

- **Opción A — correr una ejecución única (demo one-off)** para validar la orquestación  
- **Opción B — servir el flow (opcional)** para mantenerlo activo con ejecución programada


In [1]:
import prefect
print("Prefect versión:", prefect.__version__)

Prefect versión: 2.20.9


### Opción A — Corrida manual del flujo (one-off)

Esta modalidad ejecuta el flujo `etl_opensky_flow` **una sola vez**, ideal para validar
que todas las tareas del pipeline funcionan correctamente:

- extracción del snapshot dinámico  
- limpieza y particionado en Silver  
- enriquecimiento con metadatos estáticos  
- guardado final en Gold  

Se recomienda esta opción para pruebas, validación local y depuración.

### Opción A — Corrida manual del flujo (one-off)

Ejecuta el flujo `etl_opensky_flow` **una sola vez**, ideal para validar que la orquestación funciona correctamente. Permite verificar:

- que la extracción desde OpenSky responde  
- que las transformaciones Bronze → Silver → Gold se encadenan sin errores  
- que el flujo persiste los datos en el Data Lake como se espera  

Esta modalidad es la recomendada para pruebas locales y depuración antes de activar cualquier programación automática.


In [2]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

In [3]:
import importlib

# Importa el módulo de orquestación
etl = importlib.import_module("etl_opensky_flow")

# Ejecuta el flujo ETL de forma local (modo recomendado dentro del Notebook)
etl.etl_opensky_flow()

# Nota:
# También puede ejecutarse desde la terminal:
#     python src/etl_opensky_flow.py
#
# O desde el notebook:
#     !python ../src/etl_opensky_flow.py
#
# Todas las opciones ejecutan exactamente el mismo flow.

18:22:49.487 | INFO    | prefect.engine - Created flow run 'misty-agouti' for flow 'etl-opensky-full-pipeline'

18:22:49.487 | INFO    | Flow run 'misty-agouti' - View at https://app.prefect.cloud/account/1513bf29-3686-40b8-9dbf-c85ba6a6f8c0/workspace/377710aa-6343-48a1-a52d-8fd9be6fbba7/flow-runs/flow-run/06928c12-9224-73ba-8000-2363ba53ddc1

18:22:50.186 | INFO    | Flow run 'misty-agouti' - Created task run 'task_extract_aircraft_metadata-0' for task 'task_extract_aircraft_metadata'

18:22:50.186 | INFO    | Flow run 'misty-agouti' - Executing 'task_extract_aircraft_metadata-0' immediately...

18:23:29.327 | INFO    | Task run 'extract-aircraft-metadata' - Finished in state Completed()

...

💾 Datos guardados en Delta Lake: data/etl_datalake/gold/api_opensky


18:23:55.302 | INFO    | Task run 'save-gold-states' - Finished in state Completed()

✅ ETL OpenSky ejecutado correctamente.


18:23:55.800 | INFO    | Flow run 'misty-agouti' - Finished in state Completed('All states completed.')

[Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `bool`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `bool`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `dict`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersis